In [1]:
# import torch
import numpy as np
import json
# import scipy.special as sp

# import pickle as pkl
# import zlib
# import base64

In [2]:
import sys
# sys.path.append('/home/kutulu/projects/code-of-kutulu-client')
sys.path.append('..')

In [3]:
from src.envs.agents.dummy_agent import DummyAgent

In [4]:
from src.envs.league.agent_description import AgentDescription
from experiments.run_experiment import get_agent_info
from src.envs.agents.agent_factory import get_agent

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from src.envs.distance import find_path
from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS
from src.game.template import MOVE_REL_POS, REL_POSITIONS
from src.envs.agent_validator import AgentValidator
from src.envs.trainer import Trainer

In [6]:
import os
os.environ['KUTULU_ARTIFACT_OUTPUT']='../../output'

In [7]:
# class Metrics:
#     def __init__(self, use_challenge=True):
#         self.agent_validator = AgentValidator(EXTENDED_KUTULU_ACTIONS)
#         self.agent_validator_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))
#         self.competitors = None
#         if use_challenge:
#             competitors = {}
#             for competitor_type, competitor_config, new_experiment, legacy_encoder in [
#                 ('ppo', '05abe073de06428e896fcd880c9f3eac', True, True),
#                 ('qdn_conv', '20250622-045641', False, False),
#             ]:
#                 agent_info = get_agent_info(competitor_config, new_experiment=new_experiment)
#                 agent_info['legacy_encoder'] = legacy_encoder
#                 competitors[competitor_type] = agent_info
#             self.competitors = {
#                 competitor_name: get_agent(agent_info)
#                 for competitor_name, agent_info in competitors.items()
#             }

#     def _play_round(self, agents, league_level=4, num_envs=1):
#         assert len(agents) == 4
#         trainer = Trainer(
#             num_experiments=1, agents_info=None, agents=agents, shuffle=True,
#             league_level=league_level, verbose=False, seed=17,
#             silent=True, num_envs=num_envs, only_train=False, use_tqdm=False,
#         )
#         result = trainer.play_single_rollout(only_eval=True)
#         # return result
#         scores = np.array([np.argwhere(~np.isnan(result[:,i])).max().item() for i in range(4)])
#         return scores

#     def _calculate_metrics(self, agent, use_challenge=True):
#         metrics = {}
#         av = self.agent_validator
#         av_plan = self.agent_validator_plan
#         metrics['check_exp'] = \
#             av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, env_types=('normal', 'coridor', 'corner'))
#         metrics['check_wan'] = \
#             av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=('normal', 'coridor', 'corner'))
        
#         metrics['check_slsh'] = \
#             av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, env_types=('normal',))
        
#         metrics['check_exp_normal_plan1'] = av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=1, n_max=2)
#         metrics['check_exp_normal_plan0'] = av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=3, n_max=3)
        
#         metrics['acc_weighted'] = sum([
#             0.4 * metrics['check_exp'][0],
#             0.1 * metrics['check_exp_normal_plan0'][0],
#             0.1 * metrics['check_exp_normal_plan1'][0],
#             0.4 * metrics['check_wan'][0],
#         ])
#         metrics['acc_weighted_full'] = sum([
#             0.3 * metrics['check_exp'][0], # 0-1
#             0.2 * metrics['check_wan'][0], # 0-1
#             0.3 * metrics['check_slsh'][0], # 0-1
#             0.05 / 4 * metrics['check_exp'][2], # 1-4
#             0.05 / 4 * metrics['check_wan'][2], # 1-4
#             0.05 * metrics['check_exp_normal_plan0'][0], # 0-1
#             0.05 * metrics['check_exp_normal_plan1'][0], # 0-1
#         ])
#         if self.competitors is not None:
#             for competitor_type, competitor in self.competitors.items():
#                 agents = [agent, competitor, competitor, competitor]
#                 n_exps = 20
#                 n_wins = 0
#                 for _ in range(n_exps):
#                     scores = self._play_round(agents)
#                     if scores[0] == np.max(scores):
#                         n_wins += 1
#                 metrics[f'winner_score_{competitor_type}'] = n_wins / n_exps
#         return metrics

In [17]:

# seed = 12345
# maze_name = "Pac man"

# # First run - set numpy random seed before creating agents
# np.random.seed(42)
# agents1 = [
#     DummyAgent('closest', len(EXTENDED_KUTULU_ACTIONS), train=False)
#     for _ in range(4)
# ]

# trainer1 = Trainer(
#     num_experiments=1,
#     agents_info=None,
#     agents=agents1,
#     shuffle=False,  # Important: no shuffling of agent positions
#     league_level=3,
#     seed=42,  # Fixed seed for trainer
#     silent=True,
#     only_train=False,
#     use_tqdm=False,
# )

# result1 = trainer1.play_single_rollout(
#     seed=seed,
#     maze_name=maze_name,
#     only_eval=True
# )

# # Second run - reset numpy random seed before creating agents
# np.random.seed(42)
# agents2 = [
#     DummyAgent('closest', len(EXTENDED_KUTULU_ACTIONS), train=False)
#     for _ in range(4)
# ]

# trainer2 = Trainer(
#     num_experiments=1,
#     agents_info=None,
#     agents=agents2,
#     shuffle=False,
#     league_level=3,
#     seed=42,
#     silent=True,
#     only_train=False,
#     use_tqdm=False,
# )

# result2 = trainer2.play_single_rollout(
#     seed=seed,
#     maze_name=maze_name,
#     only_eval=True
# )

# # Verify identical results
# assert result1.shape == result2.shape, \
#     f"Results have different shapes: {result1.shape} vs {result2.shape}"

# assert np.array_equal(result1, result2, equal_nan=True), \
#     "Results are not identical - reproducibility failed!"

# # Additional check: verify results are not all zeros/nans
# assert not np.all(np.isnan(result1)), "Result1 is all NaN"
# assert not np.all(np.isnan(result2)), "Result2 is all NaN"
# assert result1.shape[0] > 0, "No steps were recorded"


In [22]:
from src.envs.agent_metrics import Metrics
from src.envs.agents.raw_euristic_agent import RawEuristicAgent

In [19]:
competitors = Metrics.load_default_competitors()

In [30]:
%%time
metrics = Metrics(use_challenge=True, competitors=competitors)
# metrics = Metrics(use_challenge=False)
np.random.seed(42)
# agent = DummyAgent('closest', len(EXTENDED_KUTULU_ACTIONS), train=True)
agent = RawEuristicAgent(
        state_type='raw',
        action_space_n=5,  # UP, RIGHT, DOWN, LEFT, WAIT
        train=True,
        verbose=False
    )
metrics._calculate_metrics(agent)

CPU times: user 21.6 s, sys: 3.3 s, total: 24.9 s
Wall time: 26.6 s


{'check_exp': (1.0, 0.0, 4, 0.0, 0.0),
 'check_wan': (0.625, 0.0, 1, 0.0, 0.0),
 'check_slsh': (0.5, 0.0, 2, 0.0, 0.0),
 'check_exp_normal_plan1': (1.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan0': (1.0, 0.0, 4, 0.0, 0.0),
 'acc_weighted': 0.85,
 'acc_weighted_full': 0.7375,
 'winner_score_ppo': 0.0,
 'winner_score_qdn_conv': 0.05}

In [32]:
%%time
metrics = Metrics(use_challenge=True, competitors=competitors)
# metrics = Metrics(use_challenge=False)
np.random.seed(42)
# agent = DummyAgent('closest', len(EXTENDED_KUTULU_ACTIONS), train=True)
agent = RawEuristicAgent(
        state_type='raw',
        action_space_n=5,  # UP, RIGHT, DOWN, LEFT, WAIT
        train=True,
        verbose=False
    )
metrics._calculate_metrics(agent)

CPU times: user 21.7 s, sys: 3.38 s, total: 25.1 s
Wall time: 27.4 s


{'check_exp': (1.0, 0.0, 4, 0.0, 0.0),
 'check_wan': (0.625, 0.0, 1, 0.0, 0.0),
 'check_slsh': (0.5, 0.0, 2, 0.0, 0.0),
 'check_exp_normal_plan1': (1.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan0': (1.0, 0.0, 4, 0.0, 0.0),
 'acc_weighted': 0.85,
 'acc_weighted_full': 0.7375,
 'winner_score_ppo': 0.0,
 'winner_score_qdn_conv': 0.05}

In [20]:
%%time
metrics = Metrics(use_challenge=True, competitors=competitors)
np.random.seed(42)
agent = DummyAgent('closest', len(EXTENDED_KUTULU_ACTIONS), train=True)
metrics._calculate_metrics(agent)

CPU times: user 26.7 s, sys: 4.05 s, total: 30.7 s
Wall time: 33.2 s


{'check_exp': (1.0, 0.0, 4, 0.0, 0.0),
 'check_wan': (0.625, 0.0, 1, 0.0, 0.0),
 'check_slsh': (0.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan1': (1.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan0': (1.0, 0.0, 4, 0.0, 0.0),
 'acc_weighted': 0.85,
 'acc_weighted_full': 0.5875,
 'winner_score_ppo': 0.0,
 'winner_score_qdn_conv': 0.0}

In [12]:
%%time
# metrics = Metrics(use_challenge=False)
np.random.seed(42)
agent = DummyAgent('closest', len(EXTENDED_KUTULU_ACTIONS), train=True)
metrics._calculate_metrics(agent)

CPU times: user 26.4 s, sys: 3.96 s, total: 30.4 s
Wall time: 33.6 s


{'check_exp': (1.0, 0.0, 4, 0.0, 0.0),
 'check_wan': (0.625, 0.0, 1, 0.0, 0.0),
 'check_slsh': (0.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan1': (1.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan0': (1.0, 0.0, 4, 0.0, 0.0),
 'acc_weighted': 0.85,
 'acc_weighted_full': 0.5875,
 'winner_score_ppo': 0.0,
 'winner_score_qdn_conv': 0.0}

In [53]:
metrics._calculate_metrics(agent)

{'check_exp': (1.0, 0.0, 4, 0.0, 0.0),
 'check_wan': (0.5625, 0.0, 1, 0.0, 0.0),
 'check_slsh': (0.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan1': (1.0, 0.0, 4, 0.0, 0.0),
 'check_exp_normal_plan0': (1.0, 0.0, 4, 0.0, 0.0),
 'acc_weighted': 0.825,
 'acc_weighted_full': 0.5750000000000001,
 'winner_score_ppo': 0.0,
 'winner_score_qdn_conv': 0.0}

In [33]:
agent.train

False

In [12]:
av = AgentValidator(EXTENDED_KUTULU_ACTIONS)
av_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))

In [13]:
print(av.check_entity_nearby(agent, 'WANDERER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, verbose=False))

(1.0, 0.0, 4, 0.0, 0.0)
(0.0, 0.0, 4, 0.0, 0.0)
(1.0, 0.0, 4, 0.0, 0.0)


In [14]:
# av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=2, verbose=True, env_types=['normal', 'coridor', 'corner'])
av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=2, verbose=True, env_types=['normal'])

answer: {0}, action: 0, explorers: [(0, -2)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#...1...#
#..#^#..#
#...0...#
#..#.#..#
#.......#
#..#.#..#
#########


answer: {1}, action: 1, explorers: [(2, 0)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#...0^1.#
#..#.#..#
#.......#
#..#.#..#
#########


answer: {2}, action: 2, explorers: [(0, 2)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#...0...#
#..#^#..#
#...1...#
#..#.#..#
#########


answer: {3}, action: 3, explorers: [(-2, 0)], wanderers: []
action: True, std: 0.0
#########
#..#.#..#
#.......#
#..#.#..#
#.1^0...#
#..#.#..#
#.......#
#..#.#..#
#########




([True, True, True, True], [0.0, 0.0, 0.0, 0.0], [0, 1, 2, 3])

In [10]:
self = av
entity_kind = 'EXPLORER'
# n_min, n_max = 

In [11]:
max_dist = len(self.normal_env.map) // 2
h, w = len(self.normal_env.map), len(self.normal_env.map[0])

In [12]:
explorers_list = []
for i in range(h):
    for j in range(w):
        x = self.player_pos[0] + i
        y = self.player_pos[1] + j
        if self.normal_env.map[i][j] != '#':
            explorers_list.append([(j - self.player_pos[1], i - self.player_pos[0])])

In [13]:
env = self.normal_env

In [14]:
action_list = []
output_list = []
for explorers in explorers_list:
    self._set_env(env, agent, explorers, wanderers=[], slashers=[])
    output = agent.inference_step(0)
    action = output['action']
    output_list.append(output)
    action_list.append(action)

In [15]:
_map = [list(x) for x in env.map]
for explorers, action in zip(explorers_list, action_list):
    explorer = explorers[0]
    x, y = self.player_pos[0] + explorer[0], self.player_pos[1] + explorer[1]
    _map[y][x] = str(action)
_map = [''.join(x) for x in _map]

In [16]:
_map

['#########',
 '#33#0#00#',
 '#3300000#',
 '#33#0#11#',
 '#3333111#',
 '#33#2#11#',
 '#2322222#',
 '#22#2#22#',
 '#########']

In [78]:
competitor_type, competitor_config, new_experiment = ('ppo', '05abe073de06428e896fcd880c9f3eac', True)
legacy_encoder = True
agent = get_agent_by_params(competitor_type, competitor_config, new_experiment, legacy_encoder)
print(av.check_entity_nearby(agent, 'WANDERER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, verbose=False))

(1.0, 0.0, 1, 0.0, 0.0)
(0.0, 0.0, 1, 0.0, 0.0)
(1.0, 0.0, 4, 0.0, 0.0)


In [79]:
competitor_type, competitor_config, new_experiment = ('qdn_conv', '20250622-045641', False)
legacy_encoder = False
agent = get_agent_by_params(competitor_type, competitor_config, new_experiment, legacy_encoder)
print(av.check_entity_nearby(agent, 'WANDERER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=3, verbose=False))
print(av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3, verbose=False))

(1.0, 0.0, 1, 0.0, 0.0)
(0.875, 0.0, 1, 0.0, 0.0)
(1.0, 0.0, 4, 0.0, 0.0)
